# Method 3 -- LLM-as-Judge

Two backends are available:

- **Groq (free, default)** -- `llama-3.3-70b-versatile` via Groq's free API (get a key at https://console.groq.com). No cost, so this runs directly on larger samples without a cost check first. Rate-limited to ~12,000 tokens/min on the free tier, so calls are paced with a small delay.
- **Claude (paid)** -- see the bottom of this notebook. Costs real money per call; start with a small `limit` and check actual cost before scaling up.

Both use the same prompt/rubric (`src/models/llm_judge.py::PROMPT_TEMPLATE`) so results are directly comparable.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.environ.get('GROQ_API_KEY'), 'Set GROQ_API_KEY (free at https://console.groq.com)'

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.data.loader import load_halueval_qa
from src.models.llm_judge_groq import run_llm_judge_groq
from src.evaluation.metrics import compute_metrics, print_report

ds = load_halueval_qa(num_samples=10_000)

## Run on a larger sample (free -- no cost check needed)

The free tier has **two separate limits**: a per-minute token rate (~12,000 tokens/min, paced below via `sleep_between_calls`) and a **daily token quota (100,000 tokens/day)**. A run of a few hundred judge calls can exhaust the daily quota outright -- in testing, a 300-example run hit it at 260/300. `run_llm_judge_groq()` catches this (`stop_on_rate_limit=True`, the default) and returns whatever it completed instead of raising and losing the whole run -- check `len(judge_result.predictions)` against `limit` below to see if it was cut short. To cover the full 10,000-example set, spread the run across multiple days (re-run with an increasing `limit`, or track already-judged indices) or use a paid Groq tier.

In [ ]:
judge_result = run_llm_judge_groq(ds, limit=60, sleep_between_calls=3.2)
n = len(judge_result.predictions)
print(f'Completed {n}/60 examples' + ('' if n == 60 else ' (cut short by rate limit)'))

print_report(judge_result.labels, judge_result.predictions,
             title=f'LLM-as-Judge (Groq, llama-3.3-70b-versatile, n={n})')

for r, p, t in list(zip(judge_result.rationales, judge_result.predictions, judge_result.labels))[:5]:
    print(f'pred={p} true={t} :: {r[:150]}')

In [ ]:
n = len(judge_result.predictions)
title = f'LLM-as-Judge (Groq, llama-3.3-70b-versatile, n={n})'
with open('../outputs/tables/llm_judge_groq_results.json', 'w') as f:
    json.dump({title: compute_metrics(judge_result.labels, judge_result.predictions).as_dict()}, f, indent=2)

---
## Alternative: Claude (paid)

**This path calls a paid API.** Every cell below costs money and requires `ANTHROPIC_API_KEY`. Start with a small `limit` (as below) before scaling up -- do not run the full 10,000-example set without first checking the cost of a small batch.

In [ ]:
assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY before running this section'

In [ ]:
from src.models.llm_judge import run_llm_judge

claude_result = run_llm_judge(ds, limit=20)
print_report(claude_result.labels, claude_result.predictions,
             title='LLM-as-Judge (Claude, n=20 smoke test)')